# Logging (`logging` Module)

The `logging` module is Python's built-in framework for recording information about the execution of a program.

It is used to record:

- Application events
- Errors
- Warnings
- Debug information
- System activity

Unlike `print()`, logs can be written to the console, files, or external monitoring systems.

## Why Not Just Use `print()`?

Suppose you're writing a standard data processing script using standard print tracking hooks:

```python
print("Reading CSV...")
print("Cleaning Data...")
print("Connecting to Database...")
print("Uploading Data...")
```

While this approach works during manual local development, consider real-world production constraints:
* The script runs on an automated loop every single hour.
* It parses and transforms over **5 million records** per execution sweep.
* It operates headlessly on a **remote server** with no visual display monitor.
* It suddenly crashes silently at **2:30 AM**.

When an error occurs, generic screen printouts provide zero context. **`print()` does not provide enough information** to debug automated systems.


### Real-World Analogy: Medical Records
Imagine a hospital system. A doctor does not simply write a single high-level note saying: 
> *"Patient visited."*

Instead, every single critical physiological milestone is meticulously recorded chronologically:
* **09:00 AM** → Patient admitted
* **09:15 AM** → Blood sample collected
* **09:40 AM** → Diagnosis completed
* **10:00 AM** → Medicine administered

These dense, timeline-tracked records allow any doctor to look back and accurately diagnose exactly what happened. **Logging works the same way for software**. It builds an ongoing history of events during program execution.


## What Can Logging Record?

Logging creates a structured record of your application's active health metrics and operations, such as:

* **Lifecycle Hooks**: `Application Started`, `Database Connected`
* **Network Transactions**: `API Request Received`, `User Logged In`
* **Data Processing Operations**: `File Loaded`, `Order Processed`
* **System Health Alerts**: `Data Validation Failed`, `Memory Warning`, `Unexpected Exception`

Instead of throwing temporary plain text up on a terminal screen, logging structures this behavioral data so it can be routed into searchable analytics pipelines or saved directly onto long-term storage drives.


## Your First Logger

To begin recording structured information, pull in Python's native `logging` engine and fire a basic tracking alert method:

```python
import logging

logging.warning("This is a warning")
```

### Log String Architecture Breakdown
When executed, the statement prints a standardized string pattern to the output stream:
```text
WARNING:root:This is a warning
```

* **`WARNING`**: The **Log Level**. This parameter tells you how urgent, critical, or important the recorded message is.
* **`root`**: The **Logger Name**. Because we have not initialized a custom isolated subsystem tracker yet, Python automatically uses its default baseline tracking profile named `root`.
* **`This is a warning`**: The actual **Message** payload text that we explicitly wrote.


In [1]:
# Executing your first live logger entry
import logging

# Standard Jupyter notebooks configure root loggers automatically.
# Let's fire a warning message to verify the raw structural format layout.
logging.warning("This is a warning")


| Level    | Purpose                               |
| -------- | ------------------------------------- |
| DEBUG    | Detailed information for developers   |
| INFO     | Normal program execution              |
| WARNING  | Something unexpected but not critical |
| ERROR    | An operation failed                   |
| CRITICAL | Serious error; application may stop   |


## Logging Levels in Action

Each logging level is designed to capture a specific type of operational event during program execution.

* **`DEBUG`**: Used while developing or debugging to track intricate system steps.
  ```python
  logging.debug("Loading configuration...")
  # Examples: "Reading CSV", "Entering Function X", "API Response Received"
  ```
* **`INFO`**: Represents standard, successful application lifecycle events.
  ```python
  logging.info("Employee data loaded successfully.")
  # Examples: "Server started", "File uploaded", "User logged in"
  ```
* **`WARNING`**: Indicates something unusual happened, but the application can safely continue running.
  ```python
  logging.warning("Missing salary column. Using default value.")
  ```
* **`ERROR`**: Emitted when a specific operation or requested task fails completely.
  ```python
  logging.error("Database connection failed.")
  ```
* **`CRITICAL`**: Indicates a severe application failure where the system usually cannot continue execution.
  ```python
  logging.critical("Unable to start application.")
  ```


## Why Do Only `WARNING` and Above Appear?

If you try to run all five logging levels out of the box using a fresh Python interpreter session:

```python
import logging

logging.debug("Debug")
logging.info("Info")
logging.warning("Warning")
logging.error("Error")
logging.critical("Critical")
```

### Output Behavior
```text
WARNING:root:Warning
ERROR:root:Error
CRITICAL:root:Critical
```

### The Missing Logs
Notice that `DEBUG` and `INFO` completely vanish from the output. This happens because **by default, Python enforces `WARNING` as its minimum logging threshold**. Any event severity below this floor value is silently ignored.


## Overriding Thresholds via Configuration

You can change this default behavior using the built-in configuration controller: **`logging.basicConfig()`**.

```python
import logging

logging.basicConfig(level=logging.DEBUG)
logging.debug("Debug Message")
```

### The Minimum Threshold Hierarchy
The severity framework acts as an upward waterfall filter. When you declare a target level, Python enables that specific category and *all* tiers sitting above it:

```text
 [ CRITICAL ]  (Highest)
      ▲
  [ ERROR ]
      ▲
 [ WARNING ]   <-- Default Python Floor
      ▲
   [ INFO ]
      ▲
  [ DEBUG ]    (Lowest)
```

### Threshold Resolution Profiles
* **Profile A**: Configured to `logging.INFO`
  * `DEBUG` ❌ *Hidden* | `INFO` ✅ *Shown* | `WARNING` ✅ *Shown* | `ERROR` ✅ *Shown* | `CRITICAL` ✅ *Shown*
* **Profile B**: Configured to `logging.ERROR`
  * `DEBUG` ❌ *Hidden* | `INFO` ❌ *Hidden* | `WARNING` ❌ *Hidden* | `ERROR` ✅ *Shown* | `CRITICAL` ✅ *Shown*


## Strategic Engineering Value

Managing log density via configuration parameters lets you control system behavior without modifying your core source logic:

* **Development Environment**: Set your threshold to `DEBUG`. You see every low-level movement, file parse, and query execution.
* **Production Environment**: Set your threshold to `ERROR` or `CRITICAL`. This reduces logging overhead, keeps logs clean, and ensures you are only alerted when something needs immediate attention.


In [2]:
import logging
from importlib import reload

# CRITICAL NOTE FOR JUPYTER NOTEBOOKS: 
# The logging.basicConfig() function can only be called ONCE per runtime session.
# If the root logger is already initialized, subsequent calls are ignored.
# To demonstrate the change cleanly inside a notebook cell, we reload the library:
reload(logging)

# Configure the runtime threshold to show everything from INFO and above
logging.basicConfig(level=logging.INFO)

print("--- Testing Threshold-Filtered Execution Streams ---")
logging.debug("This will be HIDDEN (below INFO threshold)")
logging.info("This will be SHOWN (matches threshold)")
logging.warning("This will be SHOWN (above threshold)")


INFO:root:This will be SHOWN (matches threshold)


--- Testing Threshold-Filtered Execution Streams ---


# Professional Logging

In professional applications, logging is not limited to printing messages on the terminal.

Logs are typically:

- Saved to files
- Formatted with timestamps
- Organized using custom loggers
- Written to multiple destinations
- Used for debugging production issues

This makes it possible to understand what happened even after the program has finished running.

## Why Save Logs to a File?

Imagine an automated **ETL batch job** configured to execute every night at **2:00 AM**. The program suddenly crashes at **2:37 AM** on a headless remote cloud server when no one is actively monitoring a terminal screen.

If you rely solely on terminal output markers:
```python
print("Connecting to Database...")
```
The text stream vanishes entirely the exact millisecond the host OS runtime process exits. 

By saving logs directly to a physical persistent file instead, you capture a historical record that can be audited later to pinpoint precisely where the failure occurred:

### Structured File Simulation (`app.log`)
```text
2026-07-26 02:00:01 INFO Job Started
2026-07-26 02:05:12 INFO Reading employee.csv
2026-07-26 02:07:10 ERROR Database Connection Failed
```


## Routing Log Streams to a File

To redirect your application metrics away from the screen and directly into storage, assign the `filename` parameter within your configuration driver:

```python
import logging

logging.basicConfig(
    filename="app.log",
    level=logging.INFO
)

logging.info("Application Started")
logging.warning("Low Memory")
logging.error("Database Connection Failed")
```

### Resulting Filesystem Behavior
When executed, **nothing is printed to your console terminal**. Instead, Python programmatically constructs a file named `app.log` in your active working directory populated with standard structured records:

```text
INFO:root:Application Started
WARNING:root:Low Memory
ERROR:root:Database Connection Failed
```


## Implementing Custom Log Formatters

The default plain layout lacks the context needed to debug complex production environments. You can extend this format by declaring token designators via the `format` configuration attribute:

```python
import logging

logging.basicConfig(
    filename="app.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Application Started")
```

### Formatted Output Archetype
```text
2026-07-26 13:17:15,102 - INFO - Application Started
```

Every incoming logging record is now automatically stamped with three essential diagnostic metadata anchors:
* **Time**: Precise down to the millisecond (`%(asctime)s`).
* **Log Level**: The exact severity indicator string (`%(levelname)s`).
* **Message**: The actual descriptive payload text string (`%(message)s`).


In [3]:
import logging
import os
from importlib import reload
from pathlib import Path

# Wipe and reload the logging subsystem to bypass Jupyter's single-call block
reload(logging)

# Target log output destination
log_file_target = Path("app.log")
if log_file_target.exists():
    log_file_target.unlink()

# Configure the system to write structured log arrays directly to disk
logging.basicConfig(
    filename=str(log_file_target),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Populate the historical file log lines
logging.info("Application context initialized.")
logging.warning("Resource constraint warning: High payload threshold crossed.")
logging.error("Data ingestion connection dropped.")

print(f"✅ Operations complete. Reading written logging contents from: {log_file_target.resolve()}\n")
print(log_file_target.read_text())


✅ Operations complete. Reading written logging contents from: E:\python-for-data-engineering\Professional Python\app.log

2026-07-26 13:19:26,449 - INFO - Application context initialized.
2026-07-26 13:19:26,450 - WARNING - Resource constraint warning: High payload threshold crossed.
2026-07-26 13:19:26,450 - ERROR - Data ingestion connection dropped.



| Placeholder     | Meaning             |
| --------------- | ------------------- |
| `%(asctime)s`   | Current date & time |
| `%(levelname)s` | Log level           |
| `%(message)s`   | Log message         |
| `%(filename)s`  | File name           |
| `%(lineno)d`    | Line number         |
| `%(name)s`      | Logger name         |


## Why Create a Custom Logger?

Until now, we have relied exclusively on high-level procedural tracking declarations:
```python
logging.info(...)
```
This forces all outputs to stream through Python's default baseline tracker called **`root`**. Production engineering environments isolate their logging sub-systems by constructing explicit, named logger instances:

```python
import logging

logger = logging.getLogger(__name__)
logger.info("Application Started")
```

### Decoupled Sub-System Mappings
Imagine a real-world analytics application structured across discrete domain-specific tasks:
```text
project/
├── database.py   --> Instantiates its own named logger namespace
├── api.py        --> Instantiates its own named logger namespace
└── pipeline.py   --> Instantiates its own named logger namespace
```
By isolating lookups, your structural runtime outputs explicitly track which precise architectural component generated a specific transaction line, simplifying system diagnostics.


## Deep Dive: The `__name__` Variable & Multi-Destination Handlers

### Automated Namespace Resolution
The built-in dual-underscore special variable **`__name__`** evaluates directly to the active module's naming framework at compilation runtime:
* If evaluated inside `database.py`, `__name__` automatically resolves to the string `"database"`.
* If evaluated inside `pipeline.py`, `__name__` automatically resolves to the string `"pipeline"`.

This ensures each component automatically registers its own clean, isolated diagnostic namespace without manual string hardcoding.

### Diverging Output via Handlers
In professional systems, we often need logs to stream to multiple destinations simultaneously:

```text
               ┌──► Console Output (Terminal View)
[ Program ] ──► [ Logger ] ──► Handlers Engine ──┤
                                                 └──► app.log File (Long-Term Storage)
```

This multi-directional routing is achieved using **Handlers**. Handlers accept a single logging record payload and split it dynamically across multiple destination sinks at once.


## Advanced Exception Traceback Capturing

When an unexpected exception occurs inside a try-except block, relying on a basic string printout ruins visibility because it destroys the error stack context:

```python
# Avoid this pattern
try:
    x = 10 / 0
except Exception as e:
    print(e)  # Only outputs: "division by zero"
```

Instead, invoke **`logging.exception()`**. This tracking function writes your custom alert string and automatically extracts and appends the complete filesystem crash **Traceback**:

```python
try:
    x = 10 / 0
except Exception:
    logging.exception("An unexpected error occurred")
```

### Resulting Output Topography
```text
ERROR:root:An unexpected error occurred
Traceback (most recent call last):
  File "<stdin>", line 2, in <module>
ZeroDivisionError: division by zero
```


In [4]:
import logging
from importlib import reload

# Reset tracking pipeline back to terminal view
reload(logging)
logging.basicConfig(level=logging.ERROR)

print("--- Testing Live Exception Traceback Harvesting ---")
try:
    # Trigger a mathematical runtime error
    calculation_anomaly = 550 / 0
except ZeroDivisionError:
    # exception() automatically pulls down the full stack context
    logging.exception("Pipeline pipeline processing halted due to an evaluation fault.")


ERROR:root:Pipeline pipeline processing halted due to an evaluation fault.
Traceback (most recent call last):
  File "C:\Users\manya\AppData\Local\Temp\ipykernel_20792\1910994681.py", line 11, in <module>
    calculation_anomaly = 550 / 0
                          ~~~~^~~
ZeroDivisionError: division by zero


--- Testing Live Exception Traceback Harvesting ---


## Logging Best Practices Matrix

| Level | Ideal Use Cases | Practical Examples |
| :--- | :--- | :--- |
| **`DEBUG`** | Intricate parameters and diagnostic data blocks. | `Function entered`, `SQL query string generated` |
| **`INFO`** | General high-level pipeline status updates. | `Job Started`, `CSV Loaded`, `User Logged In` |
| **`WARNING`** | Anomalies detected but system can recover. | `Missing optional column`, `Low disk space` |
| **`ERROR`** | Task failures that break transactions. | `Database connection timed out`, `API request failed` |
| **`CRITICAL`**| Total system infrastructure collapse. | `Configuration file missing`, `System service unbootable` |

### Data Engineering Scenario Study
Consider the historical log timeline tracking an operational enterprise data pipeline:
```text
02:00:00 INFO Job Started
02:00:05 INFO Reading employee.csv
02:00:15 INFO Loaded 2,500,000 rows
02:01:10 INFO Cleaning Missing Values
02:02:30 INFO Connecting to PostgreSQL
02:02:40 ERROR Connection Timed Out
02:02:41 INFO Retrying Connection
02:02:50 INFO Connected Successfully
02:05:20 INFO Data Uploaded
02:05:22 INFO Job Completed
```

If business stakeholders ask: *"Why did yesterday's data extraction pipeline take significantly longer than usual?"*—you can easily audit the timeline to prove exactly how many seconds were lost during the PostgreSQL connection dropout, entirely removing guesswork.


## Professional Logging Checklist

### 1. Persistent Storage
Ensure logs are routed to physical non-volatile disk space:
```python
logging.basicConfig(filename="app.log")
```

### 2. Establish Threshold Floors
Lock the application down to clean operational thresholds:
```python
level=logging.INFO
```

### 3. Chronological Metadata Formatting
Inject clean temporal and structural markers into every line:
```python
format="%(asctime)s - %(levelname)s - %(message)s"
```

### 4. Component Scope Namespaces
Isolate diagnostic lines by avoiding the global root variable:
```python
logger = logging.getLogger(__name__)
```

### 5. Automated Crash Harvesting
Never swallow exceptions; extract the entire stack traceback:
```python
logging.exception("Alert context message")
```
